<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Notification в C#, который будет представлять уведомления
пользователям. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

Требования к базовому классу Notification:
• Атрибуты: ID уведомления (NotificationId), Текст уведомления (MessageText),
Тип уведомления (Type).
• Методы:
o DisplayNotification(): метод для отображения уведомления
пользователю.
o SendNotification(): метод для отправки уведомления.
o GetNotificationDetails(): метод для получения деталей уведомления.

Требования к производным классам:
1. EmailУведомление (EmailNotification): Должно содержать дополнительные
атрибуты, такие как Адрес электронной почты (EmailAddress).
Метод SendNotification() должен быть переопределен для отправки
уведомления по электронной почте.
2. SMSУведомление (SMSNotification): Должно содержать дополнительные
атрибуты, такие как Номер телефона (PhoneNumber).
Метод SendNotification() должен быть переопределен для отправки
уведомления через SMS.
3. PushУведомление (PushNotification) (если требуется третий класс): Должно
содержать дополнительные атрибуты, такие как Платформа (Platform,
например, iOS или Android). Метод DisplayNotification() должен быть
переопределен для отображения уведомления на мобильной платформе.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;


public delegate void NotificationHandler(Notification notification);


public class Notification
{

    public int NotificationId {get; set; }
    public string MessageText {get; set; }
    public string Type {get; set; }
    public bool IsRead {get; set; }
    public List<string> Tags {get; set; }

    public event NotificationHandler NotificationRead;

    public Notification(int notificationId, string messageText, string type)
    {
        NotificationId = notificationId;
        MessageText = messageText;
        Type = type;
        IsRead = false; 
        Tags = new List<string>();
    }

    public virtual void DisplayNotification()
    {
        Console.WriteLine($"{MessageText}.");
    }

    public virtual void SendNotification()
    {
        Console.WriteLine("Уведомление отправлено.");
    }

    public virtual void GetNotificationDetails()
    {
        Console.WriteLine("Детали уведомления:");
        Console.WriteLine($"ID: {NotificationId}.");
        Console.WriteLine($"Тип: {Type}.");
        Console.WriteLine($"Прочитано: {IsRead}.");
        Console.WriteLine($"Теги: {string.Join(", ", Tags)}");
    }

    public void AddTag(string tag)
    {
        Tags.Add(tag);
    }

    public void MarkAsRead()
    {
        IsRead = true;
        NotificationRead?.Invoke(this);
    }
}

class EmailNotification : Notification
{
    public string EmailAddress {get; set; }
    public string Subject {get; set; }
    public bool HasAttachment {get; set; }

    
    public EmailNotification(string emailAddress, int notificationId, string messageText, string type) : base(notificationId, messageText, type)
    {
        this.EmailAddress = emailAddress;
        this.Subject = "Без темы";
        this.HasAttachment = false;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"Email отправлен на {EmailAddress}: {Subject}.");
    }

    public void AddAttachment()
    {
        HasAttachment = true;
        Console.WriteLine("Вложение добавлено к email.");
    }
}

class SMSNotification : Notification
{
    public string PhoneNumber {get; set; }
    private string Operator {get; set; }
    private int smsLength {get; set; }

    public SMSNotification(string phoneNumber, int notificationId, string messageText, string type) : base(notificationId, messageText, type)
    {
        this.PhoneNumber = phoneNumber;
        this.Operator = "Неизвестный оператор";
        UpdateSmsLength();
    }

    public override void SendNotification()
    {
        Console.WriteLine($"SMS отправлено на номер {PhoneNumber} через {Operator}.");
    }

    private void UpdateSmsLength()
    {
        smsLength = MessageText.Length;
    }

    public void SetOperator(string newOperator)
    {
        Operator = newOperator;
        Console.WriteLine($"Оператор изменен на: {Operator}.");
    }
}

class PushNotification : Notification
{
    public string Platform {get; set; }
    public string DeviceId {get; set; }

    public PushNotification(string platform, int notificationId, string messageText, string type) : base(notificationId, messageText, type)
    {
        this.Platform = platform;
        this.DeviceId = "Неизвестное устройство";
    }
    public override void DisplayNotification()
    {
         Console.WriteLine($"Push-уведомление на {Platform} (устройство: {DeviceId}).");
    }
}

public class NotificationManager
{
    private List<Notification> notifications;
    public event NotificationHandler MassNotificationSent;

    public NotificationManager()
    {
        notifications = new List<Notification>();
    }

    public void AddNotification(Notification notification)
    {
        notifications.Add(notification);
        Console.WriteLine($"Уведомление #{notification.NotificationId} добавлено");
    }

    public void RemoveNotification(int id)
    {
        var notification = notifications.FirstOrDefault(n => n.NotificationId == id);
        if (notification != null)
        {
            notifications.Remove(notification);
            Console.WriteLine($"Уведомление #{id} удалено");
        }
    }

    public void DisplayAllNotifications()
    {
        Console.WriteLine("\n=== Все уведомления ===");
        foreach (var notification in notifications)
        {
            notification.DisplayNotification();
        }
    }

    public void SendMassNotification()
    {
        Console.WriteLine("\n=== Массовая рассылка ===");
        foreach (var notification in notifications)
        {
            notification.SendNotification();
            MassNotificationSent?.Invoke(notification);
        }
    }

    public List<Notification> FindNotificationsByTag(string tag)
    {
        return notifications.Where(n => n.Tags.Contains(tag)).ToList();
    }

    public void MarkAllAsRead()
    {
        foreach (var notification in notifications)
        {
            notification.MarkAsRead();
        }
    }
}

static void OnNotificationRead(Notification notification)
{
    Console.WriteLine($"Пользователь прочитал уведомление #{notification.NotificationId}");
}

static void OnMassNotificationSent(Notification notification)
{
    Console.WriteLine($"Массовая рассылка: обработано #{notification.NotificationId}");
}

var manager = new NotificationManager();

    var email = new EmailNotification("test@mail.com", 1, "Ваш заказ готов", "Email");
    var sms = new SMSNotification("+79991234567", 2, "Здравствуйте!", "SMS");
    var push = new PushNotification("iOS", 3, "Новое сообщение", "Push");

    email.Subject = "Важное уведомление";
    email.AddTag("Важное");
    email.AddTag("Заказ");

    sms.SetOperator("МегаФон");
    sms.AddTag("Приветствие");

    push.DeviceId = "iPhone13,4";
    push.AddTag("Сообщение");

    // Подписка на события
    email.NotificationRead += OnNotificationRead;
    manager.MassNotificationSent += OnMassNotificationSent;

    // Добавление в менеджер
    manager.AddNotification(email);
    manager.AddNotification(sms);
    manager.AddNotification(push);

    // Демонстрация работы
    manager.DisplayAllNotifications();
    manager.SendMassNotification();
        
    manager.MarkAllAsRead();

    var importantNotifications = manager.FindNotificationsByTag("Важное");
    Console.WriteLine($"\nНайдено важных уведомлений: {importantNotifications.Count}");

Оператор изменен на: МегаФон.
Уведомление #1 добавлено
Уведомление #2 добавлено
Уведомление #3 добавлено

=== Все уведомления ===
Ваш заказ готов.
Здравствуйте!.
Push-уведомление на iOS (устройство: iPhone13,4).

=== Массовая рассылка ===
Email отправлен на test@mail.com: Важное уведомление.
Массовая рассылка: обработано #1
SMS отправлено на номер +79991234567 через МегаФон.
Массовая рассылка: обработано #2
Уведомление отправлено.
Массовая рассылка: обработано #3
Пользователь прочитал уведомление #1

Найдено важных уведомлений: 1
